In [3]:
import common
from db import get_engine
engine = get_engine()

Microsoft SQL Server 2022 (RTM-CU27-GDR) (KB5126934) - 16.0.
Logged in as: datacube_analytics
     name
0  bronze
1    gold
2  silver


In [4]:
airlines_raw = common.pd.read_csv('/workspaces/Flight-delay-analysis-E2E/data/Flight/airlines.csv')


In [5]:
airports_raw = common.pd.read_csv('/workspaces/Flight-delay-analysis-E2E/data/Flight/airports.csv')


In [6]:
flights_raw = common.pd.read_parquet('/workspaces/Flight-delay-analysis-E2E/data/Flight/flights.parquet')

In [7]:
flights_raw.head(5)

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,...,408.0,-22.0,0,0,<NA>,NaN,NaN,NaN,NaN,NaN
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,...,741.0,-9.0,0,0,<NA>,NaN,NaN,NaN,NaN,NaN
2,2015,1,1,4,US,840,N171US,SFO,CLT,20,...,811.0,5.0,0,0,<NA>,NaN,NaN,NaN,NaN,NaN
3,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,...,756.0,-9.0,0,0,<NA>,NaN,NaN,NaN,NaN,NaN
4,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,...,259.0,-21.0,0,0,<NA>,NaN,NaN,NaN,NaN,NaN


In [14]:
import json
from pathlib import Path
import pandas as pd
import requests

# 1. Qovluq və fayl yolunun təyini
output_dir = Path("/workspaces/Flight-delay-analysis-E2E/data/api")
output_dir.mkdir(parents=True, exist_ok=True)
single_json_path = output_dir / "weather_2015_all.json"

# 2. airports.csv faylını oxumaq
csv_path = "/workspaces/Flight-delay-analysis-E2E/data/Flight/airports.csv"
airports_df = pd.read_csv(csv_path)


# Sütun adlarını tapan funksiya
def find_column(df, possible_names):
  for col in df.columns:
    if col.lower() in [p.lower() for p in possible_names]:
      return col
  return None


iata_col = find_column(
    airports_df, ["iata_code", "iata", "airport_code", "code", "faa"]
)
lat_col = find_column(airports_df, ["latitude", "lat"])
lon_col = find_column(airports_df, ["longitude", "lon", "lng"])

# 3. Bütün hava limanlarının məlumatlarını toplayacaq vahid siyahı
all_airports_weather = []

print("Məlumatlar API-dən toplanılır, zəhmət olmasa gözləyin...")

for index, row in airports_df.iterrows():
  airport_code = str(row[iata_col]).strip()
  lat = row[lat_col]
  lon = row[lon_col]

  if pd.isna(lat) or pd.isna(lon) or not airport_code or airport_code == "nan":
    continue

  url = (
      f"https://archive-api.open-meteo.com/v1/archive?"
      f"latitude={lat}&longitude={lon}&start_date=2015-01-01&end_date=2015-12-31&"
      f"hourly=temperature_2m,precipitation,wind_speed_10m,visibility,"
      f"surface_pressure,wind_direction_10m,snowfall,wind_gusts_10m,weather_code"
  )

  response = requests.get(url)
  if response.status_code == 200:
    raw_data = response.json()
    hourly = raw_data.get("hourly", {})
    times = hourly.get("time", [])

    for i, t in enumerate(times):
      all_airports_weather.append({
          "airport_code": airport_code,
          "observation_time": t,
          "temperature": hourly.get("temperature_2m", [None])[i],
          "precipitation": hourly.get("precipitation", [None])[i],
          "wind_speed": hourly.get("wind_speed_10m", [None])[i],
          "visibility": hourly.get("visibility", [None])[i],
          "weather_condition": hourly.get("weather_code", [None])[i],
          "surface_pressure": hourly.get("surface_pressure", [None])[i],
          "wind_direction": hourly.get("wind_direction_10m", [None])[i],
          "snowfall": hourly.get("snowfall", [None])[i],
          "wind_gusts": hourly.get("wind_gusts_10m", [None])[i],
          "Latitude": lat,
          "Longitude": lon,
      })

    print(f"Toplandı: {airport_code}")
  else:
    print(f"Xəta ({airport_code}): Status {response.status_code}")

# 4. Bütün toplanmış məlumatları TƏK bir JSON faylına yazmaq
with open(single_json_path, "w", encoding="utf-8") as f:
  json.dump(all_airports_weather, f, ensure_ascii=False, indent=4)

print(
    f"\nUğurla tamamlandı! Bütün məlumatlar tək bir fayla yazıldı:"
    f" {single_json_path}"
)

Məlumatlar API-dən toplanılır, zəhmət olmasa gözləyin...
Toplandı: ABE
Toplandı: ABI
Toplandı: ABQ
Toplandı: ABR
Toplandı: ABY
Toplandı: ACK
Toplandı: ACT
Toplandı: ACV
Toplandı: ACY


ReadTimeout: HTTPSConnectionPool(host='archive-api.open-meteo.com', port=443): Read timed out. (read timeout=None)